# L4a: Graph and Tree Representations

A graph is a set of objects, called vertices, and a set of pairwise connections between them, called edges. Roads between cities, reactions between chemical species, and calls between functions are all graphs. A tree is the simplest connected graph: it has just enough edges to reach every vertex and no cycle. Today we define the vocabulary, look at the common graph families, and see how a graph is stored.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Describe a graph and its basic measures:__ Define vertices, edges, paths, and connectivity, and tell a directed graph from an undirected one. Compute the degree of a vertex and the density of a graph from the vertex and edge counts.
> * __Recognize complete graphs, bipartite graphs, and trees:__ State what makes each family special: every pair joined, edges only between two groups, or connected with no cycle. Explain what the edge count and the coloring or path properties of each family tell an algorithm.
> * __Choose a storage representation:__ Read a graph from an edge list into an adjacency list and an adjacency matrix. Choose between the two from the density of the graph and from whether the algorithm mostly checks for a single edge or mostly loops over the neighbors of a vertex.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

This lecture needs nothing beyond the course environment: [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) supplies the checks we run on the storage example, and the course package supplies the graph functions we call, which live in [the `GraphRepresentation.jl` file](../../../code/src/GraphRepresentation.jl). The edge list we read is in the `data` folder of this meeting.
___

## Simple Graphs
A simple graph $\mathcal{G} = (\mathcal{V},\mathcal{E})$ has a set of vertices $\mathcal{V}$ and a set of edges $\mathcal{E}$. Each edge joins two different vertices, and no edge is repeated, so there are no self-loops and no parallel edges. In a directed graph, one edge each way between two vertices counts as two different edges. An edge can carry a weight, such as a distance or a cost, or carry no weight at all.

Edges can also have a direction:
* In a __directed__ graph, each edge points from one vertex to another. In a social network, a directed edge can mean that person A follows person B, while B need not follow A.
* In an __undirected__ graph, an edge has no direction, and the connection reads the same from both ends.

A __path__ from $v_{i}$ to $v_{j}$ is a sequence of distinct vertices that starts at $v_{i}$ and ends at $v_{j}$, where each consecutive pair is joined by an edge. In a directed graph, each edge on the path must be followed in its direction. A graph is __connected__ if there is a path between every pair of vertices; for a directed graph we ignore the edge directions when asking this.

<div>
    <center>
        <img src="figs/Fig-General-Graph-Schematic.svg" width="980" alt="Left: an undirected graph with six vertices and seven weighted edges, with the degree of vertex 4 marked. Right: the same vertices and edges drawn as a directed acyclic graph, with the in-degree of vertex 2 and the out-degree of vertex 4 marked."/>
    </center>
</div>

The figure shows the same six vertices and seven edges drawn twice, once undirected and once directed, and marks the degree counts we define next.
___

## Graph Properties and Metrics
A few counts describe how connected a graph is, both at a single vertex and overall.

### Vertex degree
The __degree__ of a vertex $v_{i}\in\mathcal{V}$, written $\deg(v_{i})$, is the number of edges that touch it. In a directed graph we count the two directions separately:
* __In-degree__ $\deg^{\text{in}}(v_{i})$: the number of edges that point into $v_{i}$.
* __Out-degree__ $\deg^{\text{out}}(v_{i})$: the number of edges that point out of $v_{i}$.

The total degree of a vertex in a directed graph is $\deg(v_{i}) = \deg^{\text{in}}(v_{i}) + \deg^{\text{out}}(v_{i})$. In the figure above, vertex 4 has degree 2 in the undirected drawing, and in the directed drawing vertex 2 has in-degree 2 and vertex 4 has out-degree 1.

> __Handshaking lemma:__
>
> In any graph $\mathcal{G} = (\mathcal{V},\mathcal{E})$, the degrees add up to twice the number of edges:
> $$\sum_{v_{i} \in \mathcal{V}} \deg(v_{i}) = 2|\mathcal{E}|$$
> Each edge has two ends, and each end adds one to the degree of the vertex it touches. In a directed graph the in-degrees alone add up to $|\mathcal{E}|$, and so do the out-degrees.

### Degree summaries and density
Three numbers summarize the degrees of a whole graph. The __minimum degree__ $\delta(\mathcal{G})$ and the __maximum degree__ $\Delta(\mathcal{G})$ are the smallest and largest vertex degrees. The __average degree__ follows from the handshaking lemma:
$$\bar{d}(\mathcal{G}) = \frac{1}{|\mathcal{V}|} \sum_{v_{i} \in \mathcal{V}} \deg(v_{i}) = \frac{2|\mathcal{E}|}{|\mathcal{V}|}$$
A graph is __regular__ of degree $r$ when every vertex has degree $r$.

How many edges can a simple graph have? With $n = |\mathcal{V}|$ vertices, an undirected graph can join each of the $\binom{n}{2} = n(n-1)/2$ pairs once. A directed graph can hold an edge in each direction between a pair, so its ceiling is $n(n-1)$.

> __Graph density:__
>
> The density $\rho(\mathcal{G})$ is the fraction of the possible edges that are present:
> $$\rho(\mathcal{G}) = \frac{|\mathcal{E}|}{n(n-1)/2}\quad\text{(undirected)}\qquad\qquad \rho(\mathcal{G}) = \frac{|\mathcal{E}|}{n(n-1)}\quad\text{(directed)}$$
> A __dense__ graph has $\rho$ close to 1, and a __sparse__ graph has $\rho$ close to 0. A graph with fewer than two vertices has no possible edges, and we take its density to be 0, which is also what the course code returns. The distinction decides how a graph should be stored, which we take up next.

### Other structural parameters
Four more parameters describe the shape of a graph rather than its degrees. The diameter needs the graph to be connected; the other three are defined for any undirected graph.

* __Diameter__ $\text{diam}(\mathcal{G})$: the largest distance between any two vertices, where the distance is the number of edges on a shortest path. It is the worst case for getting from one vertex to another.
* __Clique number__ $\omega(\mathcal{G})$: the size of the largest set of vertices in which every pair is joined, such as the largest group of people who all know each other.
* __Chromatic number__ $\chi(\mathcal{G})$: the fewest colors needed so that no edge joins two vertices of the same color, such as the fewest time slots that schedule a set of classes with no student in two classes at once.
* __Independence number__ $\alpha(\mathcal{G})$: the size of the largest set of vertices with no edge among them, such as the most activities that can run at the same time with no conflict.

We will meet each of these again in the graph families that follow. Next, let's look at how a graph is stored.
___

## How are Graphs Stored?
An algorithm needs the graph in memory in a form it can query. Three representations are common: the edge list, the adjacency matrix, and the adjacency list.

### Edge list
An __edge list__ is the simplest form: one record per edge, holding the source vertex, the target vertex, and the weight if there is one. It is how a graph is usually written to a file, and it is what we read in the example at the end of this section. It is a poor form to compute on, because finding the neighbors of a vertex means scanning every record.

### Adjacency matrix
An __adjacency matrix__ $\mathbf{A}$ for a graph with $|\mathcal{V}|$ vertices is a $|\mathcal{V}|\times|\mathcal{V}|$ matrix. The entry $a_{ij}$ in row $i$ and column $j$ describes the edge from $v_{i}$ to $v_{j}$:
* __Unweighted__: $a_{ij}=1$ if the edge exists and $a_{ij}=0$ if it does not.
* __Weighted__: $a_{ij}=w_{ij}$, the weight of the edge, if the edge exists, and $a_{ij}=0$ if it does not.

For an undirected graph the matrix is symmetric, $a_{ij} = a_{ji}$; for a directed graph it need not be. The weighted form has one blind spot: a stored zero could mean either a missing edge or an edge of weight zero, so a graph with zero-weight edges needs a separate marker for missing edges.

### Adjacency list
An __adjacency list__ is a dictionary with one entry per vertex. The entry for $v_{i}$ holds the set $\mathcal{C}_{i}$ of vertices that $v_{i}$ is joined to. In a directed graph these are the vertices its edges point to, the out-neighbors. When weights are needed, they are stored beside each neighbor.

### Which representation should you use?
The choice comes down to density and to the operations the algorithm performs most.

> __Space:__
>
> An adjacency matrix takes $O(|\mathcal{V}|^{2})$ space whatever the edge count. An adjacency list takes $O(|\mathcal{V}| + |\mathcal{E}|)$ space. For a sparse graph, where $|\mathcal{E}| \ll |\mathcal{V}|^{2}$, the list is far smaller; for a dense graph the two are comparable.

The two forms also answer different questions quickly. A matrix tells whether there is an edge from $v_{i}$ to $v_{j}$ in constant time, $O(1)$, by reading one entry, but listing the neighbors of $v_{i}$ means scanning a whole row, $O(|\mathcal{V}|)$. A list gives the neighbors of $v_{i}$ directly in $O(\deg(v_{i}))$ time, but checking for one particular edge means searching that neighbor list, also $O(\deg(v_{i}))$.

So use a matrix for a dense graph or an algorithm that repeatedly tests single edges, and use a list for a sparse graph or an algorithm that repeatedly walks vertex neighborhoods. The traversal algorithms in the next lab do the latter, which is why they work from an adjacency list.

### Reading one graph into all three forms
The file `SimpleGraph.txt` in the `data` folder of this meeting is an edge list for a small directed graph with six vertices and seven weighted edges. The next lab uses a copy of the same file. We read it with [the `read_weighted_edges(...)` function](../../../code/src/GraphRepresentation.jl), then build the two computational forms with [the `adjacency_list(...)` function](../../../code/src/GraphRepresentation.jl) and [the `adjacency_matrix(...)` function](../../../code/src/GraphRepresentation.jl).

The adjacency list keeps only the out-neighbors of each vertex and drops the weights; the matrix keeps the weights. Both functions take the vertex set from the edge endpoints, so a vertex with no edges would not appear. The cell stores the edge records in `edge_records::Vector{<:NamedTuple}`, one `(source, target, weight)` record per edge, the adjacency list in `adjacency::Dict{Int64, Vector{Int64}}`, and the matrix with its vertex order in `matrix_representation::NamedTuple`.

In [ ]:
edge_records, adjacency, matrix_representation = let
    edge_path = joinpath(CHEME5800_L4A_DATA, "SimpleGraph.txt")
    records = read_weighted_edges(edge_path)
    list = adjacency_list(records)
    matrix = adjacency_matrix(records)
    records, list, matrix
end;

Let's look at the two forms side by side. The `vertex_order` entry says which vertex each row and column of the matrix belongs to.

In [ ]:
(adjacency = adjacency, vertex_order = matrix_representation.vertex_ids, matrix = matrix_representation.matrix)

Do we see what we expect? Vertex 1 points to vertices 2 and 3, and row 1 of the matrix holds the weights 10 and 100 in columns 2 and 3. Vertex 6 has an empty list because no edge leaves it.

Now let's count what each form stores. [The `representation_report(...)` function](../../../code/src/GraphRepresentation.jl) returns the vertex and edge counts, the directed density $|\mathcal{E}|/(|\mathcal{V}|(|\mathcal{V}|-1))$, the number of matrix entries $|\mathcal{V}|^{2}$, and the number of adjacency-list entries $|\mathcal{V}| + |\mathcal{E}|$, counting one dictionary key per vertex and one neighbor slot per edge. The same cell also sizes both forms for 100,000 vertices with ten edges each, at 8 bytes per entry and ignoring container overhead. The cell stores the report in `representation::NamedTuple`.

In [ ]:
representation = let
    report = representation_report(edge_records)
    n, k = 100_000, 10                          # vertices, edges per vertex
    bytes_per_entry = sizeof(Float64)
    matrix_gigabytes = n^2 * bytes_per_entry / 1e9
    adjacency_list_megabytes = (n + n * k) * bytes_per_entry / 1e6
    println("Large sparse graph: matrix ≈ $(matrix_gigabytes) GB, adjacency list ≈ $(adjacency_list_megabytes) MB")
    report
end

So what do we see? For six vertices the matrix holds 36 entries against 13 for the list, a small gap. For the large sparse graph the matrix would need about 80 gigabytes against about 9 megabytes for the list, which is the whole argument for adjacency lists on sparse graphs. The checks confirm the counts, the density, and two entries we can read off the edge list by hand.

In [ ]:
@testset "graph representations" begin
    @test representation.vertices == 6
    @test representation.edges == 7
    @test representation.density ≈ 7 / 30
    @test adjacency[1] == [2, 3]
    @test matrix_representation.matrix[1, 2] == 10.0
    @test representation.matrix_entries == 36
    @test representation.adjacency_list_entries == 13
end

___

## Complete Graphs
A __complete graph__ $K_{n}$ is a simple undirected graph on $n$ vertices in which every pair of distinct vertices is joined by an edge. It is the densest simple graph on $n$ vertices, the opposite extreme from a tree.

### Properties
The measures from the previous section all take their extreme values on $K_{n}$:
* __Edge count__: every one of the $\binom{n}{2} = n(n-1)/2$ pairs is joined, so $|\mathcal{E}| = n(n-1)/2$.
* __Degree__: every vertex touches the other $n-1$ vertices, so $K_{n}$ is regular of degree $n-1$.
* __Density__: every possible edge is present, so $\rho(K_{n}) = 1$ for $n \geq 2$.
* __Diameter__: every pair is adjacent, so $\text{diam}(K_{n}) = 1$ for $n \geq 2$.
* __Clique, chromatic, and independence numbers__: the whole vertex set is one clique, so $\omega(K_{n}) = n$; every vertex needs its own color, so $\chi(K_{n}) = n$; and no two vertices are non-adjacent, so $\alpha(K_{n}) = 1$.

### Where complete graphs appear
Complete graphs turn up wherever everything interacts with everything else, and as a worst case for many graph algorithms.

> __Uses of complete graphs:__
>
> * __Round-robin tournaments__: every team plays every other team once, so the teams and games form $K_{n}$.
> * __Worst-case inputs__: a complete graph has the most edges a simple graph can have, so many graph algorithms reach their worst running time on it, and it is a standard stress test.
> * __Optimization problems__: the traveling salesman problem is usually posed on a complete graph, where every city can be reached directly from every other and the task is to pick the cheapest tour.

The next family is the opposite in one sense: it forbids edges inside each of two groups.
___

## Bipartite Graphs
A graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ is __bipartite__ if its vertices can be split into two disjoint sets $\mathcal{V}_{1}$ and $\mathcal{V}_{2}$ so that every edge joins a vertex in $\mathcal{V}_{1}$ to a vertex in $\mathcal{V}_{2}$. No edge stays inside either set. The two sets are called the __parts__ of the graph.

<div>
    <center>
        <img src="figs/Fig-Bipartite-Graph-Schematic.png" width="280" alt="A bipartite graph drawn with four vertices in a left column and eight in a right column; every edge crosses from the left column to the right column."/>
    </center>
</div>

The figure draws a bipartite graph with four vertices on the left and eight on the right. Every edge crosses between the two columns.

> __Bipartite graph theorem:__
>
> For an undirected graph $\mathcal{G}$, the following are equivalent:
> 1. $\mathcal{G}$ is bipartite.
> 2. $\mathcal{G}$ can be properly colored with two colors, so $\chi(\mathcal{G}) \leq 2$.
> 3. $\mathcal{G}$ has no cycle of odd length.
>
> Condition 2 is the definition read as a coloring: put $\mathcal{V}_{1}$ in one color and $\mathcal{V}_{2}$ in the other. Condition 3 is the one an algorithm can check.

### Complete bipartite graphs
A __complete bipartite graph__ $K_{m,n}$ has $|\mathcal{V}_{1}| = m$, $|\mathcal{V}_{2}| = n$, and every vertex of $\mathcal{V}_{1}$ joined to every vertex of $\mathcal{V}_{2}$. It has $|\mathcal{E}| = m \cdot n$ edges. Every vertex in $\mathcal{V}_{1}$ has degree $n$ and every vertex in $\mathcal{V}_{2}$ has degree $m$, so $K_{m,n}$ is regular only when $m = n$.

### Testing whether a graph is bipartite
A graph traversal can test bipartiteness by trying to two-color the graph:

> __Two-coloring test:__
>
> 1. Mark every vertex uncolored.
> 2. Pick any uncolored vertex, give it color 1, and traverse outward from it, breadth-first or depth-first.
> 3. Give each uncolored neighbor $u$ of a colored vertex $v$ the opposite color of $v$. If a neighbor already has the same color as $v$, stop: the graph is not bipartite.
> 4. When the traversal ends, return to step 2 if any vertex is still uncolored, so every connected component is tested.
> 5. If every vertex is colored with no conflict, the graph is bipartite, and the two colors are the two parts.

The test visits each vertex once and looks at each edge at most twice, once from each end, so it runs in $O(|\mathcal{V}| + |\mathcal{E}|)$ time with an adjacency list. Breadth-first and depth-first traversal are the subject of the next lab.

### Where bipartite graphs appear
Bipartite graphs model any relationship between two different kinds of thing:

> __Uses of bipartite graphs:__
>
> * __Assignment__: workers and jobs, or students and courses, with an edge for each allowed pairing. The question is usually whether everyone can be assigned at once.
> * __Recommendation__: users and items, with an edge for each purchase or rating. Items liked by similar users are candidates to recommend.
> * __Biological networks__: genes and the proteins they regulate, or species and the habitats they occupy, where interactions occur only between the two kinds.

### Matchings and Hall's theorem
A __matching__ is a set of edges that share no vertices, that is, a set of pairings in which nobody is paired twice. The assignment question above asks for a matching that covers every vertex of $\mathcal{V}_{1}$. Hall's theorem says exactly when one exists.

> __Hall's theorem:__
>
> A bipartite graph has a matching that covers every vertex of $\mathcal{V}_{1}$ if and only if every subset $S \subseteq \mathcal{V}_{1}$ has at least $|S|$ neighbors in $\mathcal{V}_{2}$.
>
> In words: any group of people on the left must together have at least as many options on the right as there are people in the group. If every worker is assigned and every job is also filled, the matching is __perfect__, which needs $|\mathcal{V}_{1}| = |\mathcal{V}_{2}|$ as well.

### Storing a bipartite graph
The two-part structure allows a smaller matrix. A __biadjacency matrix__ $\mathbf{B}$ has one row per vertex of $\mathcal{V}_{1}$ and one column per vertex of $\mathcal{V}_{2}$, with $b_{ij} = 1$ when $v_{i}\in\mathcal{V}_{1}$ is joined to $u_{j}\in\mathcal{V}_{2}$.

The biadjacency matrix has $m \times n$ entries instead of the $(m+n)^{2}$ entries of the full adjacency matrix. The full matrix holds $\mathbf{B}$, its transpose, and two all-zero blocks for edges inside a part; only $\mathbf{B}$ carries information. An adjacency list needs no special treatment: it already takes $O(|\mathcal{V}| + |\mathcal{E}|)$ space whether or not the graph is bipartite.
___

## Trees
A __tree__ $\mathcal{T} = (\mathcal{V},\mathcal{E})$ is a connected undirected graph with no cycle. That short definition fixes a lot. A tree with $n = |\mathcal{V}|$ vertices has exactly $|\mathcal{E}| = n - 1$ edges: with fewer, some vertex would be cut off, and with more, some edge would close a cycle.

Two more properties follow from the definition:
* __Unique paths__: between any two vertices there is exactly one path. A second path would join with the first to form a cycle.
* __Minimal connectivity__: removing any edge splits the tree into two pieces, and adding any edge creates exactly one cycle, made of the new edge and the one existing path between its ends.

> __Tree characterization theorem:__
>
> For an undirected graph $\mathcal{G}$ with $n$ vertices, the following are equivalent:
> 1. $\mathcal{G}$ is a tree.
> 2. $\mathcal{G}$ is connected and has exactly $n-1$ edges.
> 3. $\mathcal{G}$ has no cycle and has exactly $n-1$ edges.
> 4. $\mathcal{G}$ is connected, and removing any edge disconnects it.
> 5. $\mathcal{G}$ has no cycle, and adding any edge creates exactly one cycle.
> 6. Any two vertices of $\mathcal{G}$ are joined by exactly one path.
>
> Any one of these can serve as the definition; the others follow.

### Rooted trees
In computing, a tree is usually drawn hanging from one chosen vertex, the __root__. Every other vertex then has a __parent__, the next vertex on its path to the root, and zero or more __children__. A vertex with no children is a __leaf__. The __height__ of the tree is the number of edges on the longest path from the root to a leaf.

<div>
    <center>
        <img src="figs/Fig-General-Tree-Schematic.svg" width="880" alt="A rooted tree drawn top down: the root at height 0, branch nodes at heights 1 and 2, and leaves at heights 2 and 3, with the empty set marking the missing children of each leaf."/>
    </center>
</div>

The figure shows a rooted tree of height 3. File systems, organization charts, and the tree of calls made by a recursive function are all rooted trees.

### Trees among graphs
Every tree is a connected graph, and every connected graph $\mathcal{G}$ contains at least one __spanning tree__: a tree $\mathcal{T}$ that uses every vertex of $\mathcal{G}$ and a subset of its edges. A graph with no cycle that need not be connected is a __forest__, a disjoint union of trees.

Trees matter to algorithms because their structure is so constrained. Many problems that have no known polynomial-time algorithm on general graphs, such as finding a largest independent set, can be solved in polynomial time on trees.
___

## Lab Exercises
We now have the vocabulary and the storage forms. The next step is to walk over a graph: visit every vertex that can be reached from a starting vertex, in a controlled order.

> __Review before lab time:__
>
> In lab we implement the two standard traversal orders, breadth-first search (BFS) and depth-first search (DFS), on an adjacency list of the same graph we read in this lecture. BFS keeps the vertices waiting to be visited in a queue and DFS keeps them in a stack, the two containers from L3b.

Bring your questions about adjacency lists; the lab builds directly on them.
___

## Summary
A graph is a set of vertices and a set of edges between them, and the way it is stored decides which questions an algorithm can answer quickly.

> __Key Takeaways:__
>
> * __Describe a graph by counting:__ The degree of a vertex counts its edges, and the density of a graph is the fraction of possible edges that are present. The handshaking lemma ties the two together, since the degrees add up to twice the edge count.
> * __Three families set the extremes:__ A complete graph has every possible edge, and a tree has one fewer edge than it has vertices, the fewest that keep it connected. An undirected graph is bipartite, with edges only between two groups, exactly when it has no odd cycle.
> * __Choose storage from density and the operations:__ An adjacency matrix answers whether one edge exists in constant time and suits dense graphs. An adjacency list takes space proportional to the vertex and edge counts, lists the neighbors of a vertex directly, and suits the sparse graphs that traversal algorithms walk over.

Next, in lab, we use the adjacency list to traverse a graph with breadth-first and depth-first search.
___